## Import Libraries

In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [2]:
from dotenv import load_dotenv
from pathlib import Path
from scipy.optimize import brentq
from scipy.interpolate import interp1d
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_curve
from qdrant_client import QdrantClient
from qdrant_client import models
from torch.utils.data import DataLoader, TensorDataset
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from utils.embedding_model import embedding_model
import numpy as np
import time
import torch
import psutil
from tqdm import tqdm
import wandb

import joblib
from utils.utils import sliding_windows

## Setup Training Variables

In [3]:
load_dotenv(".env")
load_dotenv("../.env")
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
load_dotenv(".env")

model_name = 'embedding_v2.1'
ratio = '80:10:10_time_split'
train_split = '80'
seeder = 42
window_len = os.getenv("WINDOW_SIZE")
stride_len = os.getenv("STRIDE")
num_batch = os.getenv("BATCH_SIZE")
num_epoch = os.getenv("EPOCHS")
margin = 0.2
wandb_name = model_name + '_train_' + train_split + '_' + str(seeder) + '_' + str(window_len) + '_' + str(stride_len) + '_b' + str(num_batch) + '_e' + str(num_epoch) + '_margin_' + str(margin)
print(wandb_name)

embedding_v2.1_train_80_42_1_1_b32_e100_margin_0.2


In [4]:
load_dotenv(".env")
BASE_PATH = os.getenv("BASE_PATH")
PREPROCESSED_PATH = os.getenv("PREPROCESSED_PATH")
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

suffix = f"{os.getenv('WINDOW_SIZE').replace('.', '')}_{os.getenv('STRIDE').replace('.', '')}"
preprocessed_dir = Path(BASE_PATH + PREPROCESSED_PATH)

X_train = np.load(preprocessed_dir / f'X_eo_train_{suffix}.npy')
y_train = np.load(preprocessed_dir / f'y_eo_train_{suffix}.npy')
X_val = np.load(preprocessed_dir / f'X_eo_val_{suffix}.npy')
y_val = np.load(preprocessed_dir / f'y_eo_val_{suffix}.npy')
X_test = np.load(preprocessed_dir / f'X_eo_test_{suffix}.npy')
y_test = np.load(preprocessed_dir / f'y_eo_test_{suffix}.npy')

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

X_train: (5232, 64, 160) y_train: (5232,)
X_val: (654, 64, 160) y_val: (654,)
X_test: (654, 64, 160) y_test: (654,)


In [5]:
# Raw cropped data is split/windowed in 01_data_preprocessing.ipynb.
# Training uses the saved train/val/test arrays loaded above.


In [6]:
wandb.login(key=os.getenv("WANDB_API_KEY"))
run = wandb.init(
    entity="chocomaltt",
    project="eeg-biometric-system",
    name=wandb_name,
    config={
        "model_name": model_name,
        "ratio": ratio,
        "train_split": train_split,
        "seeder": seeder,
        "window_len": os.getenv("WINDOW_SIZE"),
        "stride_len": os.getenv("STRIDE"),
        "num_batch": os.getenv("BATCH_SIZE"),
        "epoch": os.getenv("EPOCHS")
    },
    tags=[model_name, 'train_' + str(train_split), str(seeder), str(window_len), str(stride_len), str(num_batch), str(num_epoch), str(margin)]
)

process = psutil.Process(os.getpid())
initial_memory = psutil.virtual_memory()
wandb.log({
    "resource/logging_check": 1,
    "resource/cpu_percent": psutil.cpu_percent(interval=1),
    "resource/process_cpu_percent": process.cpu_percent(interval=None),
    "resource/memory_percent": initial_memory.percent,
    "resource/memory_used_gb": initial_memory.used / (1024 ** 3),
    "resource/process_memory_gb": process.memory_info().rss / (1024 ** 3),
})

wandb: Loading settings from /home/chocomaltt/.config/wandb/settings
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for http://localhost:8080.
wandb: Appending key for localhost:8080 to your netrc file: /home/chocomaltt/.netrc
wandb: Currently logged in as: chocomaltt to http://localhost:8080. Use `wandb login --relogin` to force relogin


In [7]:
print("Loaded split arrays from preprocessing notebook.")
print("Train labels:", np.unique(y_train, return_counts=True))
print("Val labels:", np.unique(y_val, return_counts=True))
print("Test labels:", np.unique(y_test, return_counts=True))


Loaded split arrays from preprocessing notebook.
Train labels: (array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108]), array([48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48,
       48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48,
       48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48,
       48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48,
       48, 48, 48, 48, 48,

In [8]:
X_train_t = torch.from_numpy(X_train.copy()).float()
y_train_t = torch.from_numpy(y_train.copy()).long()

X_test_t = torch.from_numpy(X_test.copy()).float()
y_test_t = torch.from_numpy(y_test.copy()).long()

X_val_t = torch.from_numpy(X_val.copy()).float()
y_val_t = torch.from_numpy(y_val.copy()).long()

train_ds = TensorDataset(X_train_t, y_train_t)
val_ds = TensorDataset(X_val_t, y_val_t)
test_ds = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(
    train_ds,
    batch_size=int(os.getenv("BATCH_SIZE")),
    shuffle=True,
    num_workers=int(os.getenv("NUM_WORKERS")),
    drop_last=True
)
val_loader = DataLoader(
    val_ds,
    batch_size=int(os.getenv("BATCH_SIZE")),
    shuffle=False,
    num_workers=int(os.getenv("NUM_WORKERS")),
    drop_last=False
)
test_loader = DataLoader(
    test_ds,
    batch_size=int(os.getenv("BATCH_SIZE")),
    shuffle=False,
    num_workers=int(os.getenv("NUM_WORKERS")),
    drop_last=False
)

In [9]:
model = embedding_model(
    in_channels=int(os.getenv("INPUT_CHANNELS")),
    num_classes=int(os.getenv("NUM_CLASSES")),
)
model.to(os.getenv("DEVICE"))

embedding_model(
  (input): Sequential(
    (0): LazyConv2d(0, 64, kernel_size=(1, 1), stride=(1, 1), padding=same)
    (1): SELU()
  )
  (conv2_temporal): Sequential(
    (0): LazyConv2d(0, 32, kernel_size=(4, 4), stride=(1, 1), padding=same)
    (1): SELU()
  )
  (batch_normalization): LazyBatchNorm2d(0, eps=32, momentum=0.1, affine=True, track_running_stats=True)
  (elu): ELU(alpha=1.0)
  (MaxPool2d): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (conv2_spatial): Sequential(
    (0): LazyConv2d(0, 64, kernel_size=(2, 2), stride=(1, 1), padding=same)
    (1): SELU()
  )
  (lstm): LSTM(2048, 128, batch_first=True)
  (dense): Sequential(
    (0): LazyLinear(in_features=0, out_features=128, bias=True)
    (1): SELU()
  )
)

In [10]:
# Pastikan DEVICE sudah di-set (GPU kalau ada, kalau nggak CPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Lakukan "Dry Run" untuk membangunkan layer Lazy
with torch.no_grad():
    # Ambil 1 sampel saja dari X_train_t (Ingat, ECG sudah kita buang)
    sample_eeg = X_train_t[:1].to(DEVICE, non_blocking=True)

    sample_eeg = sample_eeg.unsqueeze(1)
    
    # Masukkan ke model. Setelah baris ini lewat, dimensi layer Lazy resmi terbentuk!
    _ = model(sample_eeg)

# 3. Hitung Parameter
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✓ Model berjalan di: {DEVICE}")
print(f"✓ Model initialized - Total params: {total_params:,}, Trainable: {trainable_params:,}")

# 4. Cek Memori GPU (Opsional)
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.memory_allocated()/1e9:.2f}GB allocated")

✓ Model berjalan di: cuda
✓ Model initialized - Total params: 1,172,896, Trainable: 1,172,896
GPU Memory: 0.01GB allocated


/home/chocomaltt/Kuliah/eeg-biometric-system/eeg/lib/python3.10/site-packages/torch/nn/modules/conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1025.)
  return F.conv2d(


In [11]:
client = QdrantClient(url="http://localhost:6333")

if not client.collection_exists("eeg_embeddings_v2"):
    client.create_collection(
        collection_name="eeg_embeddings_v2",
        vectors_config=models.VectorParams(size=128, distance=models.Distance.COSINE),
    )

In [12]:
from pytorch_metric_learning import losses # Import library metric learning

LEARNING_RATE = float(os.getenv("LEARNING_RATE", 1e-4))
EPOCHS = int(os.getenv("EPOCHS", 100))

# 1. Ganti Loss Function menjadi Triplet Margin Loss
# Margin 0.2 adalah standar yang bagus untuk permulaan
criterion = losses.TripletMarginLoss(margin=margin)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

checkpoint_filepath = "best_eeg_embedding_model.pth"
best_val_loss = float('inf')
best_epoch = 0
patience = 10 # Kesabaran bisa dinaikkan sedikit untuk metric learning
wait = 0
best_weights = None

# History (Kita hilangkan akurasi sementara, karena akurasi embedding 
# dihitung secara terpisah nanti menggunakan KNN/Cosine Similarity)
history = {'loss': [], 'val_loss': []}

process = psutil.Process(os.getpid())
psutil.cpu_percent(interval=None)
process.cpu_percent(interval=None)

print(f"Starting Embedding Training with Early Stopping (patience={patience})...")

for epoch in range(EPOCHS):
    # --- TRAINING PHASE ---
    model.train()
    train_loss = 0.0
    
    for batch_idx, (data_eeg, targets) in enumerate(train_loader):
        if data_eeg.dim() == 3: 
            data_eeg = data_eeg.unsqueeze(1)
        data_eeg = data_eeg.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        
        optimizer.zero_grad()

        # print(f"train batch shape: {data_eeg.shape}")
        #
        # Outputnya sekarang adalah VEKTOR EMBEDDING
        embeddings = model(data_eeg) 
        
        # Triplet loss akan otomatis mencari pasangan (Anchor, Positive, Negative)
        # berdasarkan label (targets) yang kamu berikan
        loss = criterion(embeddings, targets)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for data_eeg, targets in val_loader:
            data_eeg = data_eeg.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            
            embeddings = model(data_eeg)
            loss = criterion(embeddings, targets)
            
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    # Simpan History
    history['loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)

    memory = psutil.virtual_memory()
    process_memory = process.memory_info().rss / (1024 ** 3)

    wandb.log({
        "epoch/epoch": epoch,
        "epoch/train_loss": avg_train_loss,
        "epoch/val_loss": avg_val_loss,
        "epoch/best_val_loss": best_val_loss,
        "epoch/best_epoch": best_epoch,
        "epoch/patience": patience,
        "epoch/wait": wait,
        "epoch/best_weights": best_weights,
        "epoch/checkpoint_filepath": checkpoint_filepath,
        "epoch/optimizer_state_dict": optimizer.state_dict(),
        "resource/cpu_percent": psutil.cpu_percent(interval=None),
        "resource/process_cpu_percent": process.cpu_percent(interval=None),
        "resource/memory_percent": memory.percent,
        "resource/memory_used_gb": memory.used / (1024 ** 3),
        "resource/process_memory_gb": process_memory,
    })

    print(f"Epoch {epoch+1:03d}/{EPOCHS} | Train Loss (Triplet): {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # --- CHECKPOINT & EARLY STOPPING ---
    if avg_val_loss < best_val_loss:
        print(f" -> Validation loss improved ({best_val_loss:.4f} to {avg_val_loss:.4f}). Saving model... 💾")
        best_val_loss = avg_val_loss
        best_epoch = epoch
        best_weights = model.state_dict().copy()
        wait = 0
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss,
        }, checkpoint_filepath)
    else:
        wait += 1
        
    if wait >= patience:
        print(f"\nEarly stopping triggered! No improvement for {patience} epochs.")
        if best_weights is not None:
            model.load_state_dict(best_weights)
            print(f"Restored best model weights from Epoch {best_epoch+1}.")
        break

# After full training without early stop, last epoch may not be best — always use best checkpoint
if best_weights is not None:
    model.load_state_dict(best_weights)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Training finished. Vektor biometrik siap digunakan! 🚀")

Starting Embedding Training with Early Stopping (patience=10)...


wandb: WARNING Artifact "source-eeg-biometric-system-_home_chocomaltt_Kuliah_eeg-biometric-system_notebooks_02_model_training.ipynb" already exists with the same content. No new version will be created.


Epoch 001/100 | Train Loss (Triplet): 0.2025 | Val Loss: 0.2244
 -> Validation loss improved (inf to 0.2244). Saving model... 💾
Epoch 002/100 | Train Loss (Triplet): 0.1960 | Val Loss: 0.2983
Epoch 003/100 | Train Loss (Triplet): 0.1813 | Val Loss: 0.2435
Epoch 004/100 | Train Loss (Triplet): 0.1820 | Val Loss: 0.1568
 -> Validation loss improved (0.2244 to 0.1568). Saving model... 💾
Epoch 005/100 | Train Loss (Triplet): 0.1750 | Val Loss: 0.1881
Epoch 006/100 | Train Loss (Triplet): 0.1792 | Val Loss: 0.1477
 -> Validation loss improved (0.1568 to 0.1477). Saving model... 💾
Epoch 007/100 | Train Loss (Triplet): 0.1740 | Val Loss: 0.1536
Epoch 008/100 | Train Loss (Triplet): 0.1713 | Val Loss: 0.1420
 -> Validation loss improved (0.1477 to 0.1420). Saving model... 💾
Epoch 009/100 | Train Loss (Triplet): 0.1550 | Val Loss: 0.1326
 -> Validation loss improved (0.1420 to 0.1326). Saving model... 💾
Epoch 010/100 | Train Loss (Triplet): 0.1592 | Val Loss: 0.1336
Epoch 011/100 | Train Loss (

In [13]:
# Enrollment gallery: train + val (test held out for evaluation)
model.eval()
X_enroll = np.concatenate([X_train, X_val], axis=0)
y_enroll = np.concatenate([y_train, y_val], axis=0)

emb_batch = int(os.getenv("BATCH_SIZE"))
enroll_ds = TensorDataset(
    torch.from_numpy(X_enroll).float(),
    torch.from_numpy(y_enroll).long(),
)
enroll_loader = DataLoader(
    enroll_ds,
    batch_size=emb_batch,
    shuffle=False,
    num_workers=int(os.getenv("NUM_WORKERS")),
    drop_last=False,
)

emb_chunks, label_chunks = [], []
with torch.no_grad():
    for data_eeg, targets in tqdm(enroll_loader, desc="Extract embeddings (enrollment)"):
        data_eeg = data_eeg.to(DEVICE, non_blocking=True)
        emb = model(data_eeg).cpu().numpy()
        emb_chunks.append(emb)
        label_chunks.append(targets.numpy())

embeddings_matrix = np.concatenate(emb_chunks, axis=0)
subject_ids = np.concatenate(label_chunks, axis=0)

out_path = Path(BASE_PATH + PREPROCESSED_PATH) / "embeddings_eo_train_val.npz"
out_path.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(out_path, embeddings=embeddings_matrix, subject_ids=subject_ids)
print(f"Saved local embedding backup: {out_path}  shape={embeddings_matrix.shape}")

qdrant_batch = 256
for start in tqdm(
    range(0, len(embeddings_matrix), qdrant_batch),
    desc="Upsert to Qdrant",
):
    end = min(start + qdrant_batch, len(embeddings_matrix))
    points = [
        models.PointStruct(
            id=start + i,
            vector=embeddings_matrix[start + i].tolist(),
            payload={"subject_id": int(subject_ids[start + i])},
        )
        for i in range(end - start)
    ]
    client.upsert(collection_name="eeg_embeddings_v2", points=points)

print(f"Upserted {len(embeddings_matrix)} points to collection 'eeg_embeddings_v2'.")

Extract embeddings (enrollment): 100%|██████████| 184/184 [00:02<00:00, 73.41it/s]


Saved local embedding backup: Dataset/preprocessed/embeddings_eo_train_val.npz  shape=(5886, 128)


Upsert to Qdrant: 100%|██████████| 23/23 [00:00<00:00, 24.95it/s]

Upserted 5886 points to collection 'eeg_embeddings_v2'.


# model = torch.load('../best_eeg_embedding_model.pth') <br>
coba modifikasi arsitektur model (conv 1d -> 2d) <br>
perkecil kernel size (8 -> ...) <br>
cek performance per satu subject (waktu, accuracy) <br>
cek usage cpu + memory <br>

In [19]:
# client = QdrantClient("/home/chocomaltt/Kuliah/eeg-biometric-system/qdrant_storage/collections/eeg_embeddings")
client = QdrantClient(url="http://localhost:6333")
model.eval()

# Top-1 identification only uses the nearest point. ROC/EER needs both
# genuine and impostor scores, so collect several neighbors per query.
roc_query_limit = 200

y_true = []
y_scores = []
top1_correct = 0
total_test_samples = 0

with torch.no_grad():
    for data_eeg, targets in test_loader:
        data_eeg = data_eeg.to(DEVICE, non_blocking=True)

        embeddings = model(data_eeg).cpu().numpy()
        targets = targets.numpy()

        for i in range(len(embeddings)):
            query_vector = embeddings[i].tolist()
            true_label = int(targets[i])

            search_result = client.query_points(
                collection_name="eeg_embeddings_v2",
                query=query_vector,
                limit=roc_query_limit
            )
            points = search_result.points
            if not points:
                continue

            best_match = points[0]
            predicted_label = int(best_match.payload["subject_id"])
            top1_correct += int(predicted_label == true_label)
            total_test_samples += 1

            for point in points:
                candidate_label = int(point.payload["subject_id"])
                y_true.append(1 if candidate_label == true_label else 0)
                y_scores.append(point.score)

y_true = np.array(y_true)
y_scores = np.array(y_scores)

classes, counts = np.unique(y_true, return_counts=True)
class_counts = dict(zip(classes.tolist(), counts.tolist()))
print("ROC label counts:", class_counts)

if len(classes) < 2:
    raise ValueError(
        "ROC/EER needs both genuine and impostor scores. "
        f"Got labels {class_counts}; increase roc_query_limit or check Qdrant payloads."
    )

fpr, tpr, thresholds = roc_curve(y_true, y_scores)
far = fpr
frr = 1 - tpr

# Pick the ROC threshold where FAR and FRR are closest. This avoids NaN
# interpolation when ROC points contain duplicate FPR values.
eer_idx = np.nanargmin(np.abs(far - frr))
eer = (far[eer_idx] + frr[eer_idx]) / 2
eer_threshold = thresholds[eer_idx]
top1_accuracy = top1_correct / total_test_samples

memory = psutil.virtual_memory()
process = psutil.Process(os.getpid())
# wandb.log({
#     "eval/top1_accuracy": top1_accuracy,
#     "eval/top1_accuracy_percent": top1_accuracy * 100,
#     "eval/eer": float(eer),
#     "eval/eer_percent": float(eer) * 100,
#     "eval/eer_threshold": float(eer_threshold),
#     "eval/total_test_samples": total_test_samples,
#     "eval/roc_scores": len(y_true),
#     "eval/genuine_scores": int(class_counts.get(1, 0)),
#     "eval/impostor_scores": int(class_counts.get(0, 0)),
#     "eval/roc_query_limit": roc_query_limit,
#     "resource/cpu_percent": psutil.cpu_percent(interval=None),
#     "resource/process_cpu_percent": process.cpu_percent(interval=None),
#     "resource/memory_percent": memory.percent,
#     "resource/memory_used_gb": memory.used / (1024 ** 3),
#     "resource/process_memory_gb": process.memory_info().rss / (1024 ** 3),
# })

print("\n=== HASIL EVALUASI BIOMETRIK ===")
print(f"Total Sampel Test : {total_test_samples}")
print(f"Total Skor ROC    : {len(y_true)}")
print(f"Genuine / Impostor: {class_counts.get(1, 0)} / {class_counts.get(0, 0)}")
print(f"Akurasi Top-1     : {top1_accuracy * 100:.2f}%")
print(f"EER (Makin kecil makin bagus) : {eer * 100:.2f}%")
print(f"Threshold Ideal   : {eer_threshold:.4f}")

wandb.finish()

ROC label counts: {0: 96162, 1: 34638}

=== HASIL EVALUASI BIOMETRIK ===
Total Sampel Test : 654
Total Skor ROC    : 130800
Genuine / Impostor: 34638 / 96162
Akurasi Top-1     : 98.01%
EER (Makin kecil makin bagus) : 11.69%
Threshold Ideal   : 0.8405


In [ ]:
target_subject_id = 5
threshold = eer_threshold

subject_mask = y_test == target_subject_id
X_single = X_test[subject_mask]
y_single = y_test[subject_mask]

print("Subject: ", target_subject_id)
print("Total test windows: ", len(X_single))

single_ds = TensorDataset(
    torch.from_numpy(X_single).float(),
    torch.from_numpy(y_single).long(),
)

single_loader = DataLoader(
    single_ds,
    batch_size=int(os.getenv("BATCH_SIZE")),
    num_workers=int(os.getenv("NUM_WORKERS")),
    drop_last=False,
)

correct_top1 = 0
accepted = 0
total = 0
scores = []

model.eval()

with torch.no_grad():
    for data_eeg, targets in single_loader:
        data_eeg = data_eeg.to(DEVICE, non_blocking=True)
        embeddings = model(data_eeg).cpu().numpy()
        targets = targets.numpy()

        for i in range(len(embeddings)):
            query_vector = embeddings[i].tolist()
            true_label = int(targets[i])

            result = client.query_points(
                collection_name="eeg_embeddings_v2",
                query=query_vector,
                limit=1
            )

            if not result.points:
                continue

            best_match = result.points[0]
            predicted_label = int(best_match.payload["subject_id"])
            score = best_match.score

            correct_top1 += int(predicted_label == true_label)
            accepted += int(score >= threshold)
            scores.append(score)
            total += 1

top1_acc = correct_top1 / total
accept_rate = accepted / total

print("\n=== HASIL EVALUASI BIOMETRIK ===")
print(f"Subject ID            : {target_subject_id}")
print(f"Total Test Windows   : {total}")
print(f"Correct Top-1        : {correct_top1}")
print(f"Accept Rate          : {accept_rate * 100:.2f}%")
print(f"Top-1 Accuracy       : {top1_acc * 100:.2f}%")
print(f"Accept Rate          : {accept_rate * 100:.2f}%")
print(f"Threshold            : {threshold:.4f}")
print(f"Mean Similarity  : {np.mean(scores):.4f}")
print(f"Min Similarity   : {np.min(scores):.4f}")
print(f"Max Similarity   : {np.max(scores):.4f}")

Subject:  5
Total test windows:  6

=== HASIL EVALUASI BIOMETRIK ===
Subject ID            : 5
Total Test Windows   : 6
Correct Top-1        : 6
Accept Rate          : 100.00%
Top-1 Accuracy       : 100.00%
Accept Rate          : 100.00%
Threshold            : 0.8405
Mean Similarity  : 0.9741
Min Similarity   : 0.9686
Max Similarity   : 0.9809


In [22]:
torch.save(model, wandb_name + ".pth")